# Comparative Analysis and Reproducibility

This analysis compares three stages of the project:

1. Reproduction of Farhood et al. using the original Mathematics and xAPI datasets.
2. Generalisation to the independent UCI Predict Students' Dropout and Academic Success dataset.
3. Generalisation to the newly constructed Mendeley dataset.

The comparison focuses on:

- relative model performance,
- model ranking,
- consistency between holdout and five-fold cross-validation,
- and the direction and magnitude of the Lasso feature-selection effect.

Absolute accuracy values are not treated as directly equivalent across all datasets because the prediction targets and student populations differ.

In [14]:
from pathlib import Path

import pandas as pd


project_root = Path.cwd()

output_dir = (
    project_root
    / "results"
    / "comparative_analysis"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)


results = {
    "Mathematics reproduction": {
        "Holdout": {
            "Linear Regression": (87.80, 89.46),
            "Logistic Regression": (89.95, 91.56),
            "SVM": (87.30, 89.85),
            "Decision Tree": (89.62, 89.65),
            "Random Forest": (93.58, 92.86),
            "KNN": (73.05, 82.42),
            "XGBoost": (93.49, 92.10),
        },
        "5-fold CV": {
            "Linear Regression": (87.75, 89.28),
            "Logistic Regression": (89.94, 91.36),
            "SVM": (87.59, 89.96),
            "Decision Tree": (89.94, 89.55),
            "Random Forest": (93.83, 92.96),
            "KNN": (73.44, 83.36),
            "XGBoost": (93.52, 91.93),
        },
    },

    "xAPI reproduction": {
        "Holdout": {
            "Linear Regression": (90.93, 91.49),
            "Logistic Regression": (91.04, 91.70),
            "SVM": (91.25, 91.03),
            "Decision Tree": (88.01, 88.30),
            "Random Forest": (92.36, 91.45),
            "KNN": (88.41, 90.71),
            "XGBoost": (91.66, 90.72),
        },
        "5-fold CV": {
            "Linear Regression": (91.11, 91.47),
            "Logistic Regression": (91.20, 91.66),
            "SVM": (91.36, 91.35),
            "Decision Tree": (87.95, 88.07),
            "Random Forest": (92.44, 91.27),
            "KNN": (88.37, 91.20),
            "XGBoost": (91.80, 90.21),
        },
    },

    "UCI existing dataset": {
        "Holdout": {
            "Linear Regression": (90.06, 90.83),
            "Logistic Regression": (90.26, 91.11),
            "SVM": (87.86, 90.47),
            "Decision Tree": (85.86, 86.13),
            "Random Forest": (90.61, 90.46),
            "KNN": (75.62, 85.70),
            "XGBoost": (90.69, 90.50),
        },
        "5-fold CV": {
            "Linear Regression": (90.10, 90.85),
            "Logistic Regression": (90.28, 91.09),
            "SVM": (87.81, 90.52),
            "Decision Tree": (85.76, 86.16),
            "Random Forest": (90.71, 90.49),
            "KNN": (75.63, 85.86),
            "XGBoost": (90.71, 90.46),
        },
    },

    "Constructed GPA 3.0": {
        "Holdout": {
            "Linear Regression": (87.19, 87.33),
            "Logistic Regression": (87.49, 87.08),
            "SVM": (87.23, 87.60),
            "Decision Tree": (86.55, 86.24),
            "Random Forest": (88.12, 87.31),
            "KNN": (83.19, 86.54),
            "XGBoost": (89.05, 87.97),
        },
        "5-fold CV": {
            "Linear Regression": (87.22, 86.81),
            "Logistic Regression": (87.60, 86.29),
            "SVM": (87.33, 87.39),
            "Decision Tree": (86.44, 86.02),
            "Random Forest": (88.23, 87.19),
            "KNN": (83.26, 87.67),
            "XGBoost": (88.82, 88.06),
        },
    },
}


rows = []

for dataset, evaluations in results.items():
    for evaluation, models in evaluations.items():
        for model, values in models.items():

            no_lasso, lasso = values

            rows.append(
                {
                    "dataset": dataset,
                    "evaluation": evaluation,
                    "model": model,
                    "without_lasso": no_lasso,
                    "with_lasso": lasso,
                    "lasso_change_pp": round(
                        lasso - no_lasso,
                        2
                    ),
                }
            )


comparison_df = pd.DataFrame(rows)

comparison_df.head()

,dataset,evaluation,model,without_lasso,with_lasso,lasso_change_pp
0,Mathematics reproduction,Holdout,Linear Regression,87.80,89.46,1.66
1,Mathematics reproduction,Holdout,Logistic Regression,89.95,91.56,1.61
2,Mathematics reproduction,Holdout,SVM,87.30,89.85,2.55
3,Mathematics reproduction,Holdout,Decision Tree,89.62,89.65,0.03
4,Mathematics reproduction,Holdout,Random Forest,93.58,92.86,-0.72


## Overall Model Comparison

In [2]:
comparison_table = comparison_df.pivot_table(
    index=["dataset", "evaluation", "model"],
    values=[
        "without_lasso",
        "with_lasso",
        "lasso_change_pp",
    ],
)

comparison_table

lasso_change_pp  \
dataset                  evaluation model                                  
Constructed GPA 3.0      5-fold CV  Decision Tree                  -0.42   
                                    KNN                             4.41   
                                    Linear Regression              -0.41   
                                    Logistic Regression            -1.31   
                                    Random Forest                  -1.04   
                                    SVM                             0.06   
                                    XGBoost                        -0.76   
                         Holdout    Decision Tree                  -0.31   
                                    KNN                             3.35   
                                    Linear Regression               0.14   
                                    Logistic Regression            -0.41   
                                    Random Forest                  -0.81   
                                    SVM                             0.37   
                                    XGBoost                        -1.08   
Mathematics reproduction 5-fold CV  Decision Tree                  -0.39   
                                    KNN                             9.92   
                                    Linear Regression               1.53   
                                    Logistic Regression             1.42   
                                    Random Forest                  -0.87   
                                    SVM                             2.37   
                                    XGBoost                        -1.59   
                         Holdout    Decision Tree                   0.03   
                                    KNN                             9.37   
                                    Linear Regression               1.66   
                                    Logistic Regression             1.61   
                                    Random Forest                  -0.72   
                                    SVM                             2.55   
                                    XGBoost                        -1.39   
UCI existing dataset     5-fold CV  Decision Tree                   0.40   
                                    KNN                            10.23   
                                    Linear Regression               0.75   
                                    Logistic Regression             0.81   
                                    Random Forest                  -0.22   
                                    SVM                             2.71   
                                    XGBoost                        -0.25   
                         Holdout    Decision Tree                   0.27   
                                    KNN                            10.08   
                                    Linear Regression               0.77   
                                    Logistic Regression             0.85   
                                    Random Forest                  -0.15   
                                    SVM                             2.61   
                                    XGBoost                        -0.19   
xAPI reproduction        5-fold CV  Decision Tree                   0.12   
                                    KNN                             2.83   
                                    Linear Regression               0.36   
                                    Logistic Regression             0.46   
                                    Random Forest                  -1.17   
                                    SVM                            -0.01   
                                    XGBoost                        -1.59   
                         Holdout    Decision Tree                   0.29   
                                    KNN                             2.30   
                             

## Model Rankings

In [3]:
comparison_df["rank_without_lasso"] = (
    comparison_df
    .groupby(
        ["dataset", "evaluation"]
    )["without_lasso"]
    .rank(
        ascending=False,
        method="min"
    )
)

comparison_df["rank_with_lasso"] = (
    comparison_df
    .groupby(
        ["dataset", "evaluation"]
    )["with_lasso"]
    .rank(
        ascending=False,
        method="min"
    )
)


ranking_table = comparison_df[
    [
        "dataset",
        "evaluation",
        "model",
        "without_lasso",
        "rank_without_lasso",
        "with_lasso",
        "rank_with_lasso",
    ]
].sort_values(
    [
        "dataset",
        "evaluation",
        "rank_without_lasso",
    ]
)

ranking_table

,dataset,evaluation,model,without_lasso,rank_without_lasso,with_lasso,rank_with_lasso
55,Constructed GPA 3.0,5-fold CV,XGBoost,88.82,1.0,88.06,1.0
53,Constructed GPA 3.0,5-fold CV,Random Forest,88.23,2.0,87.19,4.0
50,Constructed GPA 3.0,5-fold CV,Logistic Regression,87.60,3.0,86.29,6.0
51,Constructed GPA 3.0,5-fold CV,SVM,87.33,4.0,87.39,3.0
49,Constructed GPA 3.0,5-fold CV,Linear Regression,87.22,5.0,86.81,5.0
52,Constructed GPA 3.0,5-fold CV,Decision Tree,86.44,6.0,86.02,7.0
54,Constructed GPA 3.0,5-fold CV,KNN,83.26,7.0,87.67,2.0
48,Constructed GPA 3.0,Holdout,XGBoost,89.05,1.0,87.97,1.0
46,Constructed GPA 3.0,Holdout,Random Forest,88.12,2.0,87.31,4.0
43,Constructed GPA 3.0,Holdout,Logistic Regression,87.49,3.0,87.08,5.0


## Lasso Effect Comparison

In [4]:
lasso_table = comparison_df.pivot_table(
    index="model",
    columns=[
        "dataset",
        "evaluation",
    ],
    values="lasso_change_pp",
)

lasso_table.round(2)

dataset             Constructed GPA 3.0         Mathematics reproduction  \
evaluation                    5-fold CV Holdout                5-fold CV   
model                                                                      
Decision Tree                     -0.42   -0.31                    -0.39   
KNN                                4.41    3.35                     9.92   
Linear Regression                 -0.41    0.14                     1.53   
Logistic Regression               -1.31   -0.41                     1.42   
Random Forest                     -1.04   -0.81                    -0.87   
SVM                                0.06    0.37                     2.37   
XGBoost                           -0.76   -1.08                    -1.59   

dataset                     UCI existing dataset         xAPI reproduction  \
evaluation          Holdout            5-fold CV Holdout         5-fold CV   
model                                                                        
Decision Tree          0.03                 0.40    0.27              0.12   
KNN                    9.37                10.23   10.08              2.83   
Linear Regression      1.66                 0.75    0.77              0.36   
Logistic Regression    1.61                 0.81    0.85              0.46   
Random Forest         -0.72                -0.22   -0.15             -1.17   
SVM                    2.55                 2.71    2.61             -0.01   
XGBoost               -1.39                -0.25   -0.19             -1.59   

dataset                      
evaluation          Holdout  
model                        
Decision Tree          0.29  
KNN                    2.30  
Linear Regression      0.56  
Logistic Regression    0.66  
Random Forest         -0.91  
SVM                   -0.22  
XGBoost               -0.94

## GPA 3.5 Sensitivity Analysis

In [5]:
sensitivity_results = {
    "Holdout": {
        "Linear Regression": (79.79, 79.60),
        "Logistic Regression": (80.35, 80.22),
        "SVM": (82.05, 82.55),
        "Decision Tree": (84.53, 84.30),
        "Random Forest": (85.70, 86.27),
        "KNN": (77.12, 81.02),
        "XGBoost": (88.75, 88.49),
    },

    "5-fold CV": {
        "Linear Regression": (79.76, 79.57),
        "Logistic Regression": (80.29, 80.10),
        "SVM": (81.88, 82.26),
        "Decision Tree": (84.78, 84.30),
        "Random Forest": (85.85, 86.00),
        "KNN": (77.24, 81.86),
        "XGBoost": (88.70, 88.18),
    },
}


sensitivity_rows = []

for evaluation, models in sensitivity_results.items():
    for model, values in models.items():

        no_lasso, lasso = values

        sensitivity_rows.append(
            {
                "evaluation": evaluation,
                "model": model,
                "without_lasso": no_lasso,
                "with_lasso": lasso,
                "lasso_change_pp": round(
                    lasso - no_lasso,
                    2
                ),
            }
        )


sensitivity_df = pd.DataFrame(
    sensitivity_rows
)

sensitivity_df

,evaluation,model,without_lasso,with_lasso,lasso_change_pp
0,Holdout,Linear Regression,79.79,79.60,-0.19
1,Holdout,Logistic Regression,80.35,80.22,-0.13
2,Holdout,SVM,82.05,82.55,0.50
3,Holdout,Decision Tree,84.53,84.30,-0.23
4,Holdout,Random Forest,85.70,86.27,0.57
5,Holdout,KNN,77.12,81.02,3.90
6,Holdout,XGBoost,88.75,88.49,-0.26
7,5-fold CV,Linear Regression,79.76,79.57,-0.19
8,5-fold CV,Logistic Regression,80.29,80.10,-0.19
9,5-fold CV,SVM,81.88,82.26,0.38


## Best-Performing Models

In [8]:
best_rows = []

for (dataset, evaluation), group in comparison_df.groupby(
    ["dataset", "evaluation"],
    sort=False
):

    best_no_score = group["without_lasso"].max()
    best_lasso_score = group["with_lasso"].max()

    best_no_models = " / ".join(
        group.loc[
            group["without_lasso"] == best_no_score,
            "model"
        ].tolist()
    )

    best_lasso_models = " / ".join(
        group.loc[
            group["with_lasso"] == best_lasso_score,
            "model"
        ].tolist()
    )

    best_rows.append(
        {
            "dataset": dataset,
            "evaluation": evaluation,
            "best_without_lasso": best_no_models,
            "accuracy_without_lasso": best_no_score,
            "best_with_lasso": best_lasso_models,
            "accuracy_with_lasso": best_lasso_score,
        }
    )


best_model_summary = pd.DataFrame(best_rows)

best_model_summary

,dataset,evaluation,best_without_lasso,accuracy_without_lasso,best_with_lasso,accuracy_with_lasso
0,Mathematics reproduction,Holdout,Random Forest,93.58,Random Forest,92.86
1,Mathematics reproduction,5-fold CV,Random Forest,93.83,Random Forest,92.96
2,xAPI reproduction,Holdout,Random Forest,92.36,Logistic Regression,91.70
3,xAPI reproduction,5-fold CV,Random Forest,92.44,Logistic Regression,91.66
4,UCI existing dataset,Holdout,XGBoost,90.69,Logistic Regression,91.11
5,UCI existing dataset,5-fold CV,Random Forest / XGBoost,90.71,Logistic Regression,91.09
6,Constructed GPA 3.0,Holdout,XGBoost,89.05,XGBoost,87.97
7,Constructed GPA 3.0,5-fold CV,XGBoost,88.82,XGBoost,88.06


## Holdout versus Five-Fold Cross-Validation Consistency

Small differences between holdout and five-fold cross-validation indicate that the mean model-performance patterns were generally consistent across the two evaluation procedures.

In [9]:
holdout_df = (
    comparison_df[
        comparison_df["evaluation"] == "Holdout"
    ]
    [
        [
            "dataset",
            "model",
            "without_lasso",
            "with_lasso",
        ]
    ]
    .rename(
        columns={
            "without_lasso": "holdout_without_lasso",
            "with_lasso": "holdout_with_lasso",
        }
    )
)


cv_df = (
    comparison_df[
        comparison_df["evaluation"] == "5-fold CV"
    ]
    [
        [
            "dataset",
            "model",
            "without_lasso",
            "with_lasso",
        ]
    ]
    .rename(
        columns={
            "without_lasso": "cv_without_lasso",
            "with_lasso": "cv_with_lasso",
        }
    )
)


stability_df = holdout_df.merge(
    cv_df,
    on=["dataset", "model"]
)


stability_df["without_lasso_abs_difference_pp"] = (
    stability_df["holdout_without_lasso"]
    - stability_df["cv_without_lasso"]
).abs().round(2)


stability_df["with_lasso_abs_difference_pp"] = (
    stability_df["holdout_with_lasso"]
    - stability_df["cv_with_lasso"]
).abs().round(2)


stability_summary = (
    stability_df
    .groupby("dataset")
    [
        [
            "without_lasso_abs_difference_pp",
            "with_lasso_abs_difference_pp",
        ]
    ]
    .agg(["mean", "max"])
    .round(2)
)


stability_summary

without_lasso_abs_difference_pp        \
                                                    mean   max   
dataset                                                          
Constructed GPA 3.0                                 0.11  0.23   
Mathematics reproduction                            0.19  0.39   
UCI existing dataset                                0.05  0.10   
xAPI reproduction                                   0.11  0.18   

                         with_lasso_abs_difference_pp        
                                                 mean   max  
dataset                                                      
Constructed GPA 3.0                              0.44  1.13  
Mathematics reproduction                         0.26  0.94  
UCI existing dataset                             0.05  0.16  
xAPI reproduction                                0.26  0.51

## Comparative Lasso Summary

In [15]:
lasso_direction_df = comparison_df.copy()


lasso_direction_df["lasso_effect"] = (
    lasso_direction_df["lasso_change_pp"]
    .apply(
        lambda x:
            "Improved"
            if x > 0
            else "Reduced"
            if x < 0
            else "Unchanged"
    )
)


lasso_direction_summary = (
    lasso_direction_df
    .groupby(
        [
            "dataset",
            "model",
            "lasso_effect",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)


lasso_direction_summary

lasso_effect                                  Improved  Reduced
dataset                  model                                 
Constructed GPA 3.0      Decision Tree               0        2
                         KNN                         2        0
                         Linear Regression           1        1
                         Logistic Regression         0        2
                         Random Forest               0        2
                         SVM                         2        0
                         XGBoost                     0        2
Mathematics reproduction Decision Tree               1        1
                         KNN                         2        0
                         Linear Regression           2        0
                         Logistic Regression         2        0
                         Random Forest               0        2
                         SVM                         2        0
                         XGBoost                     0        2
UCI existing dataset     Decision Tree               2        0
                         KNN                         2        0
                         Linear Regression           2        0
                         Logistic Regression         2        0
                         Random Forest               0        2
                         SVM                         2        0
                         XGBoost                     0        2
xAPI reproduction        Decision Tree               2        0
                         KNN                         2        0
                         Linear Regression           2        0
                         Logistic Regression         2        0
                         Random Forest               0        2
                         SVM                         0        2
                         XGBoost                     0        2

## Reproducibility Against Published Results

The reproduced Mathematics and xAPI results are compared with the results reported by Farhood et al. to assess whether the principal model-performance patterns can be reproduced.

The Mathematics experiment uses the authors' original implementation with minor compatibility changes required for current software versions. The xAPI experiment follows the preprocessing and evaluation methodology described in the paper because an equivalent xAPI machine-learning script was not available in the public repository.

In [17]:
published_results = {
    "Mathematics": {
        "Holdout": {
            "Linear Regression": (87.80, 89.46),
            "Logistic Regression": (89.94, 91.54),
            "SVM": (87.30, 89.85),
            "Decision Tree": (89.62, 89.65),
            "Random Forest": (93.58, 92.87),
            "KNN": (73.05, 82.41),
            "XGBoost": (93.49, 92.10),
        },
        "5-fold CV": {
            "Linear Regression": (87.75, 89.27),
            "Logistic Regression": (89.94, 91.39),
            "SVM": (87.59, 89.98),
            "Decision Tree": (89.94, 89.49),
            "Random Forest": (93.82, 92.94),
            "KNN": (73.44, 83.48),
            "XGBoost": (93.53, 91.94),
        },
    },

    "xAPI": {
        "Holdout": {
            "Linear Regression": (90.47, 91.07),
            "Logistic Regression": (91.15, 91.34),
            "SVM": (91.53, 91.00),
            "Decision Tree": (88.69, 88.07),
            "Random Forest": (92.69, 91.46),
            "KNN": (86.10, 90.23),
            "XGBoost": (91.37, 90.54),
        },
        "5-fold CV": {
            "Linear Regression": (90.73, 91.28),
            "Logistic Regression": (91.26, 91.51),
            "SVM": (91.49, 91.12),
            "Decision Tree": (88.36, 87.96),
            "Random Forest": (92.70, 91.22),
            "KNN": (85.77, 91.01),
            "XGBoost": (91.51, 90.03),
        },
    },
}

In [18]:
reproduction_rows = []

reproduction_mapping = {
    "Mathematics": "Mathematics reproduction",
    "xAPI": "xAPI reproduction",
}

for published_dataset, reproduced_dataset in reproduction_mapping.items():

    for evaluation, models in published_results[
        published_dataset
    ].items():

        for model, published_values in models.items():

            published_no, published_lasso = published_values

            reproduced_row = comparison_df[
                (comparison_df["dataset"] == reproduced_dataset)
                & (comparison_df["evaluation"] == evaluation)
                & (comparison_df["model"] == model)
            ].iloc[0]

            reproduction_rows.append(
                {
                    "dataset": published_dataset,
                    "evaluation": evaluation,
                    "model": model,

                    "published_without_lasso": published_no,
                    "reproduced_without_lasso":
                        reproduced_row["without_lasso"],

                    "difference_without_lasso_pp": round(
                        reproduced_row["without_lasso"]
                        - published_no,
                        2
                    ),

                    "published_with_lasso": published_lasso,
                    "reproduced_with_lasso":
                        reproduced_row["with_lasso"],

                    "difference_with_lasso_pp": round(
                        reproduced_row["with_lasso"]
                        - published_lasso,
                        2
                    ),
                }
            )


reproducibility_df = pd.DataFrame(
    reproduction_rows
)

reproducibility_df

,dataset,evaluation,model,published_without_lasso,reproduced_without_lasso,difference_without_lasso_pp,published_with_lasso,reproduced_with_lasso,difference_with_lasso_pp
0,Mathematics,Holdout,Linear Regression,87.80,87.80,0.00,89.46,89.46,0.00
1,Mathematics,Holdout,Logistic Regression,89.94,89.95,0.01,91.54,91.56,0.02
2,Mathematics,Holdout,SVM,87.30,87.30,0.00,89.85,89.85,0.00
3,Mathematics,Holdout,Decision Tree,89.62,89.62,0.00,89.65,89.65,0.00
4,Mathematics,Holdout,Random Forest,93.58,93.58,0.00,92.87,92.86,-0.01
5,Mathematics,Holdout,KNN,73.05,73.05,0.00,82.41,82.42,0.01
6,Mathematics,Holdout,XGBoost,93.49,93.49,0.00,92.10,92.10,0.00
7,Mathematics,5-fold CV,Linear Regression,87.75,87.75,0.00,89.27,89.28,0.01
8,Mathematics,5-fold CV,Logistic Regression,89.94,89.94,0.00,91.39,91.36,-0.03
9,Mathematics,5-fold CV,SVM,87.59,87.59,0.00,89.98,89.96,-0.02


In [22]:
reproducibility_summary = (
    reproducibility_df
    .assign(
        abs_difference_without=lambda x:
            x["difference_without_lasso_pp"].abs(),

        abs_difference_with=lambda x:
            x["difference_with_lasso_pp"].abs(),
    )
    .groupby("dataset")
    [
        [
            "abs_difference_without",
            "abs_difference_with",
        ]
    ]
    .agg(["mean", "max"])
    .round(2)
)

reproducibility_summary

abs_difference_without       abs_difference_with      
                              mean   max                mean   max
dataset                                                           
Mathematics                   0.00  0.01                0.02  0.12
xAPI                          0.61  2.60                0.20  0.48

compact table showing the actual mean Lasso change across the two evaluation methods:

In [23]:
mean_lasso_effect = (
    comparison_df
    .groupby(
        [
            "dataset",
            "model",
        ]
    )["lasso_change_pp"]
    .mean()
    .round(2)
    .unstack("dataset")
)


mean_lasso_effect

dataset,Constructed GPA 3.0,Mathematics reproduction,UCI existing dataset,xAPI reproduction
model,,,,
Decision Tree,-0.36,-0.18,0.34,0.20
KNN,3.88,9.64,10.16,2.56
Linear Regression,-0.13,1.60,0.76,0.46
Logistic Regression,-0.86,1.52,0.83,0.56
Random Forest,-0.92,-0.80,-0.18,-1.04
SVM,0.22,2.46,2.66,-0.12
XGBoost,-0.92,-1.49,-0.22,-1.27


## Comparative Summary

### Model Performance and Ranking

The exact ranking of the seven machine-learning models was not constant across datasets. Random Forest achieved the highest reported accuracy in the Mathematics and xAPI reproductions when all features were used. On the independent UCI dataset, Random Forest and XGBoost remained among the strongest models without Lasso, while Logistic Regression achieved the highest reported accuracy after Lasso feature selection. On the constructed GPA 3.0 dataset, XGBoost achieved the highest reported accuracy under both holdout and five-fold cross-validation, with and without Lasso.

### Lasso Feature-Selection Effect

The effect of Lasso was strongly model-dependent. KNN improved after Lasso in every dataset and under both evaluation procedures. Its mean improvement across holdout and five-fold cross-validation was approximately 9.64 percentage points for Mathematics, 2.56 points for xAPI, 10.16 points for the UCI dataset, and 3.88 points for the constructed dataset. In contrast, Random Forest and XGBoost decreased after Lasso across all four datasets.

The remaining algorithms showed greater dataset dependence. Logistic Regression benefited from Lasso on Mathematics, xAPI and UCI, but declined on the constructed GPA 3.0 dataset. SVM improved substantially on Mathematics and UCI, changed very little on xAPI, and showed only a small improvement on the constructed dataset. These results support the conclusion that feature selection does not have a uniform effect across machine-learning algorithms or datasets.

### Holdout versus Five-Fold Cross-Validation Consistency

Mean accuracies from repeated holdout and five-fold cross-validation were generally close. The mean absolute difference between the two evaluation procedures was small for all datasets. This indicates that the broad model-performance conclusions were not strongly dependent on the choice between repeated holdout and five-fold cross-validation.

### Sensitivity Analysis

Changing the Higher/Lower threshold from GPA 3.0 to GPA 3.5 did not materially alter the principal conclusions from the constructed dataset. XGBoost remained the strongest-performing model under both holdout and five-fold cross-validation. KNN continued to benefit substantially from Lasso, while the effect of Lasso remained model-dependent.

### Overall Generalisation

The results support partial generalisation of the findings reported by Farhood et al. Random Forest, XGBoost and Logistic Regression repeatedly appeared among the stronger conventional machine-learning models, although the exact leading model varied across datasets. The clearest generalised finding was that the influence of Lasso was model-dependent rather than uniformly beneficial. KNN consistently improved following Lasso feature selection, whereas Random Forest and XGBoost generally showed reduced accuracy. The results therefore reproduce the main model-performance patterns of the original study while also showing that exact rankings and feature-selection effects depend on the dataset.

In [21]:
comparison_df.to_csv(
    output_dir / "final_model_comparison.csv",
    index=False
)

ranking_table.to_csv(
    output_dir / "model_rankings.csv",
    index=False
)

lasso_table.to_csv(
    output_dir / "lasso_effect_comparison.csv"
)

sensitivity_df.to_csv(
    output_dir / "gpa35_sensitivity_comparison.csv",
    index=False
)

best_model_summary.to_csv(
    output_dir / "best_model_summary.csv",
    index=False
)

stability_df.to_csv(
    output_dir / "holdout_vs_cv_consistency.csv",
    index=False
)

stability_summary.to_csv(
    output_dir / "consistency_summary.csv"
)

mean_lasso_effect.to_csv(
    output_dir / "mean_lasso_effect.csv"
)

reproducibility_df.to_csv(
    output_dir / "reproducibility_comparison.csv",
    index=False
)

reproducibility_summary.to_csv(
    output_dir / "reproducibility_summary.csv"
)

print("All comparative analysis outputs saved.")

All comparative analysis outputs saved.


# Save these summaries

In [25]:
comparison_df.to_csv(
    output_dir / "final_model_comparison.csv",
    index=False
)

ranking_table.to_csv(
    output_dir / "model_rankings.csv",
    index=False
)

lasso_table.to_csv(
    output_dir / "lasso_effect_comparison.csv"
)

sensitivity_df.to_csv(
    output_dir / "gpa35_sensitivity_comparison.csv",
    index=False
)

best_model_summary.to_csv(
    output_dir / "best_model_summary.csv",
    index=False
)

stability_df.to_csv(
    output_dir / "holdout_vs_cv_consistency.csv",
    index=False
)

stability_summary.to_csv(
    output_dir / "consistency_summary.csv"
)

lasso_direction_summary.to_csv(
    output_dir / "lasso_direction_summary.csv"
)

mean_lasso_effect.to_csv(
    output_dir / "mean_lasso_effect.csv"
)

reproducibility_df.to_csv(
    output_dir / "reproducibility_comparison.csv",
    index=False
)

reproducibility_summary.to_csv(
    output_dir / "reproducibility_summary.csv"
)

print("All comparative analysis outputs saved.")

All comparative analysis outputs saved.
